# 사전 등록 검증: 지자체 인구정책 뉴스레터의 주력 주제 고르기

- **사전 등록:** [`preregistration_niche_topic.md`](preregistration_niche_topic.md) · [`niche_candidates.py`](niche_candidates.py) — 커밋 `0f41961`, 후보 B~F 값을 받기 **전**.
- 모든 후보: 같은 범용 파이프라인, 통일 규칙 6차. 통과 = 전체·모든 규모 구간 잘못된 경보 ≤ 10%.
- 점수 = 감지율 × 커버 시군구 수(2025). 1위가 현재 주제(청년 순이동, 전입 2천+) 점수 × 1.1 이상이면 주력 주제 교체.
- 후보 A는 이미 본 자료(청년 순이동)라 해석할 때 표시합니다.

In [ ]:
import os, urllib.request, warnings; warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/quanter02/chelsea-analytics/claude/great-dirac-3wwgzy/experiments/"
for f in ["kosis_monitor.py", "niche_candidates.py", "track_record.py"]:
    if not os.path.exists(f): urllib.request.urlretrieve(RAW + f, f)
import pandas as pd, kosis_monitor as K, niche_candidates as N, track_record as TR
outs = {name: K.run(tp, rule="v6", verbose=False) for name, tp in N.CANDIDATES.items()}
base = K.run(K.TOPICS[N.BASELINE], rule="v6", verbose=False)

## 1. 공식 채점 (채점 기간 2018~2025)

In [ ]:
def gun(out, year=2025):
    tp = out["topic"]
    return sum(1 for (u, y), d in out["units"].items() if y == year and d["size"] >= tp.min_size and str(d["name"]).endswith("군"))
rows = {name: {**N.score(o), "군_커버": gun(o), "α": o["calibration"].alpha, "m": round(o["calibration"].m, 3)} for name, o in outs.items()}
S = pd.DataFrame(rows).T
b = {**N.score(base), "군_커버": gun(base)}
show = S.copy()
for c in ["감지율", "잘못된경보", "구간최대_잘못된경보", "상반기내"]: show[c] = show[c].map(lambda v: f"{float(v):.1%}")
show["점수"] = show["점수"].map(lambda v: f"{float(v):.1f}")
print(f"기준(현재 주제 {N.BASELINE}): 감지율 {b['감지율']:.1%}, 커버 {b['커버_시군구']}곳(군 {b['군_커버']}), 점수 {b['점수']:.1f}")
show

In [ ]:
choice, why = N.rank(S.assign(통과=S.통과.astype(bool), 점수=S.점수.astype(float)), float(b["점수"]))
print("결정:", why)

## 2. 규모 구간별 (작은 지역에서도 기준을 지키는지)

In [ ]:
for name, o in outs.items():
    print(name); display(o["groups"])

## 3. 적중 기록 미리 재기 (결정에 쓰지 않음)
채점 기간에 매달 발행했다면 3·6·9월호 경보가 연말 확정치로 몇 % 맞았나. 정의는 `track_record.py`.

In [ ]:
P = []
for name, o in {**outs, f"(기준) {N.BASELINE}": base}.items():
    p = TR.precision(o); p.insert(0, "주제", name); P.append(p)
pd.concat(P, ignore_index=True)

## 4. 탐색: 주제를 넘나드는 신호 (사전 등록 아님, 가설 만들기용)
2026년 실시간 경보가 여러 이동 주제에 겹치는 지역. 같은 자료(전입·전출)의 부분집합끼리라 겹치는 것이 당연한 부분도 있으므로 해석에 주의합니다.

In [ ]:
live = []
for name, o in {**outs}.items():
    L = o["live"]
    if len(L): live.append(L[L.상태 == "유지"].assign(주제=name)[["주제", "지역", "방향", "이탈률"]])
X = pd.concat(live, ignore_index=True)
piv = X.pivot_table(index="지역", columns="주제", values="방향", aggfunc="first")
piv["경보_주제수"] = piv.notna().sum(1)
piv.sort_values("경보_주제수", ascending=False).head(20)